# First step in co-reg
- Load files
    - HCR processed data (first round)
        - Manually check if the first round data was OK.
    - Cortical z-stack data
        - The last session with desired resolution
            - 400x400 for now. 
            - Try 700x700 later.
- Process files
    1. Save z-stack segmentation outline file (for QCing automatic landmarks)
    2. Save z-stack centroid (for generating automatic landmarks)
    3. Save HCR cell centroid (for generating automatic landmarks)


In [5]:
from pathlib import Path
import tifffile as tiff
import numpy as np
import cv2

from lamf_analysis.code_ocean import code_ocean_utils as cou

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
save_path = Path('/root/capsule/scratch/755252_2024-12-19_coreg_cpsam/755252_2024-12-19_seg-mask-outline.tif')
save_path.parent.mkdir(parents=True, exist_ok=True)
czstack_seg_path = '/root/capsule/data/multiplane-ophys_755252_2024-12-19_11-34-11_cortical-zstack-segmentation_2025-11-14_00-34-06/channel_0_ref_0/segmentation_masks.tif'
# hcr_path = 

In [8]:
cou.get_derived_assets_df(755252, 'HCR', data_name='HCR')

Error: 401 Client Error: Unauthorized for url: https://codeocean.allenneuraldynamics.org/api/v1/data_assets/search

Message: invalid API Key

Data:
{
  "message": "invalid API Key"
}

In [3]:
czstack_masks = tiff.imread(czstack_seg_path)
czstack_masks_outline = np.zeros_like(czstack_masks,dtype=np.uint8)
kernel = np.ones((3, 3), np.uint8)
for i_plane, plane in enumerate(czstack_masks):
    ids = np.unique(plane)
    ids = ids[ids!=0]
    for id in ids:
        # Creates a uint8 array (1/0) that OpenCV can use
        mask = (plane==id).astype(np.uint8)
        eroded_mask = cv2.erode(mask, kernel, iterations=1)
        edge = cv2.subtract(mask, eroded_mask)==1
        czstack_masks_outline[i_plane][edge] += 1

In [ ]:
tiff.imwrite(save_path, czstack_masks_outline)